In [1]:
import pandas as pd

BASE = "https://cct-ds-code-challenge-input-data.s3.af-south-1.amazonaws.com/"

sr = pd.read_csv(BASE + "sr.csv.gz", nrows=5000)

print("Shape (rows, columns):", sr.shape)
print("\nColumn names:")
print(sr.columns.tolist())
print("\nData types:")
print(sr.dtypes)
sr.head()

Shape (rows, columns): (5000, 16)

Column names:
['Unnamed: 0', 'notification_number', 'reference_number', 'creation_timestamp', 'completion_timestamp', 'directorate', 'department', 'branch', 'section', 'code_group', 'code', 'cause_code_group', 'cause_code', 'official_suburb', 'latitude', 'longitude']

Data types:
Unnamed: 0                int64
notification_number       int64
reference_number        float64
creation_timestamp          str
completion_timestamp        str
directorate                 str
department                  str
branch                      str
section                     str
code_group                  str
code                        str
cause_code_group            str
cause_code                  str
official_suburb             str
latitude                float64
longitude               float64
dtype: object


,Unnamed: 0,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude
0,0,400583534,9.109492e+09,2020-10-07 06:55:18+02:00,2020-10-08 15:36:35+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area Central,District: Blaauwberg,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Wear and tear,MONTAGUE GARDENS,-33.872839,18.522488
1,1,400555043,9.108995e+09,2020-07-09 16:08:13+02:00,2020-07-14 14:27:01+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,SOMERSET WEST,-34.078916,18.848940
2,2,400589145,9.109614e+09,2020-10-27 10:21:59+02:00,2020-10-28 17:48:15+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,STRAND,-34.102242,18.821116
3,3,400538915,9.108601e+09,2020-03-19 06:36:06+02:00,2021-03-29 20:34:19+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Bellville,TD Customer complaint groups,Paint Markings Lines&Signs,Road Markings,Wear and tear,RAVENSMEAD,-33.920019,18.607209
4,4,400568554,NaN,2020-08-25 09:48:42+02:00,2020-08-31 08:41:13+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,CLAREMONT,-33.987400,18.453760


In [2]:
#load all rows
sr = pd.read_csv(
    BASE + "sr.csv.gz",
    index_col=0,
    dtype={"notification_number": str, "reference_number": str},
)

print(sr.shape)
print(sr.dtypes)
print(sr["directorate"].value_counts())

(941634, 15)
notification_number         str
reference_number            str
creation_timestamp          str
completion_timestamp        str
directorate                 str
department                  str
branch                      str
section                     str
code_group                  str
code                        str
cause_code_group            str
cause_code                  str
official_suburb             str
latitude                float64
longitude               float64
dtype: object
directorate
WATER AND SANITATION                422834
ENERGY                              278117
URBAN WASTE MANAGEMENT               84634
FINANCE                              47909
COMMUNITY SERVICES AND HEALTH        28960
URBAN MOBILITY                       28028
HUMAN SETTLEMENTS                    25566
CORPORATE SERVICES                   13112
ECONOMIC GROWTH                       1913
SAFETY AND SECURITY                   1124
OFFICE OF THE CITY MANAGER               1
SPATIAL 

In [3]:
#check locations
no_location = sr["latitude"].isna() | sr["longitude"].isna()

print(no_location.sum(), "requests with no location")
print(round(no_location.mean() * 100, 1), "% of all requests")

has_location = sr[~no_location]
print("latitude range:", has_location["latitude"].min(), "to", has_location["latitude"].max())
print("longitude range:", has_location["longitude"].min(), "to", has_location["longitude"].max())

212364 requests with no location
22.6 % of all requests
latitude range: -34.348814637483 to -33.49232817
longitude range: 18.319446502448 to 18.96053944


In [ ]:
# Group	        Count	    What happens in section 2
# Has lat/lon	729,270	    Gets a real hex ID
# No lat/lon	212,364	    Gets 0, as the challenge says. Not a failure

In [6]:
# look at hexagon file
import geopandas as gpd

hex8 = gpd.read_file(BASE + "city-hex-polygons-8.geojson")

print(hex8.shape)
print(hex8.columns.tolist())
print(hex8.head())

print("duplicate hex ids:", hex8["index"].duplicated().sum())

(3832, 4)
['index', 'centroid_lat', 'centroid_lon', 'geometry']
             index  centroid_lat  centroid_lon  \
0  88ad361801fffff    -33.859427     18.677843   
1  88ad361803fffff    -33.855696     18.668766   
2  88ad361805fffff    -33.855263     18.685959   
3  88ad361807fffff    -33.851532     18.676881   
4  88ad361809fffff    -33.867322     18.678806   

                                            geometry  
0  POLYGON ((18.68119 -33.8633, 18.68357 -33.8592...  
1  POLYGON ((18.67211 -33.85957, 18.6745 -33.8555...  
2  POLYGON ((18.68931 -33.85914, 18.69169 -33.855...  
3  POLYGON ((18.68023 -33.85541, 18.68261 -33.851...  
4  POLYGON ((18.68215 -33.8712, 18.68454 -33.8671...  
duplicate hex ids: 0


In [7]:
#look at answer key
sr_hex = pd.read_csv(BASE + "sr_hex.csv.gz", dtype=str)

print(sr_hex.shape)
print(sr_hex.columns.tolist())

zeros = sr_hex["h3_level8_index"] == "0"
print(zeros.sum(), "requests with hex 0")

used = sr_hex.loc[~zeros, "h3_level8_index"]
print(used.nunique(), "different hexes used")
print((~used.isin(hex8["index"])).sum(), "requests in a hex not in the city file")

(941634, 16)
['notification_number', 'reference_number', 'creation_timestamp', 'completion_timestamp', 'directorate', 'department', 'branch', 'section', 'code_group', 'code', 'cause_code_group', 'cause_code', 'official_suburb', 'latitude', 'longitude', 'h3_level8_index']
212364 requests with hex 0
2082 different hexes used
3 requests in a hex not in the city file
